In [ ]:
# ============================================================
# AI REALIZATION GAP — EU27
# STEP 06 — DESCRIPTIVE ANALYSIS & GAP DIAGNOSTICS
# ============================================================

from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Realization_Gap_EU27"
)

PROCESSED_DIR = PROJECT_ROOT / "03_processed_data"
TABLE_DIR = PROJECT_ROOT / "05_tables"
FIGURE_DIR = PROJECT_ROOT / "06_figures"
LOG_DIR = PROJECT_ROOT / "08_logs"
STEP06_FIGURE_DIR = FIGURE_DIR / "Step_06_Descriptive_Analysis"

for folder in [
    TABLE_DIR,
    FIGURE_DIR,
    STEP06_FIGURE_DIR,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

In [ ]:
# ============================================================
# LOAD MASTER DATASET
# ============================================================

master_path = (
    PROCESSED_DIR /
    "Step_05_Master_AI_Realization_Gap_EU27.csv"
)

df = pd.read_csv(master_path)

print("Shape:", df.shape)
print("Countries:", df["iso3"].nunique())
print("Years:", sorted(df["year"].unique()))

display(df.head())

In [ ]:
# ============================================================
# BASIC QC
# ============================================================

print("Duplicate country-year:")
print(
    df.duplicated(
        ["iso3", "year"]
    ).sum()
)

print("\nMissing core variables:")

core_vars = [
    "ai_adoption_pct",
    "ai_potential",
    "ai_realization_gap",
    "vhcn_pct",
    "ln_gdp_pc_pps"
]

display(
    df[core_vars]
    .isna()
    .sum()
    .to_frame("missing_n")
)

In [ ]:
# REPRODUCIBILITY CHECK — source panel and essential variables
expected_years = [2021, 2023, 2024, 2025]
needed_columns = [
    "iso3", "country", "year", "ai_adoption_pct", "ict_specialists_pct",
    "hrst_pct", "hightech_emp_pct", "ai_potential", "ai_adoption_norm",
    "ai_realization_gap", "ai_realization_gap_z", "ai_realization_gap_equal",
    "vhcn_pct", "gdp_pc_pps", "ln_gdp_pc_pps"
]
missing_columns = sorted(set(needed_columns) - set(df.columns))
assert not missing_columns, f"Missing master columns: {missing_columns}"
assert len(df) == 108, f"Expected 108 observations, got {len(df)}"
assert df['iso3'].nunique() == 27, "Expected 27 EU countries"
assert sorted(df['year'].unique().tolist()) == expected_years, "Unexpected analysis years"
assert not df.duplicated(['iso3', 'year']).any(), "Duplicate country-year keys"
assert df.groupby('iso3')['year'].nunique().eq(4).all(), "Incomplete country-year panel"
assert df[needed_columns].notna().all().all(), "Missing values in Step 06 core columns"
print("Reproducibility checks passed: 27 countries, four years, 108 rows.")


In [ ]:
# ============================================================
# TABLE 1 — DESCRIPTIVE STATISTICS
# ============================================================

desc_vars = [
    "ai_adoption_pct",
    "ict_specialists_pct",
    "hrst_pct",
    "hightech_emp_pct",
    "ai_potential",
    "ai_realization_gap",
    "vhcn_pct",
    "gdp_pc_pps",
    "ln_gdp_pc_pps"
]

descriptive = (
    df[desc_vars]
    .agg([
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
)

descriptive.columns = [
    "N",
    "Mean",
    "SD",
    "Min",
    "Median",
    "Max"
]

display(descriptive.round(3))

In [ ]:
descriptive.to_csv(
    TABLE_DIR /
    "Step_06_Table1_Descriptive_Statistics.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# YEARLY SUMMARY
# ============================================================

yearly = (
    df
    .groupby("year")
    .agg(
        ai_adoption_mean=(
            "ai_adoption_pct",
            "mean"
        ),
        ai_potential_mean=(
            "ai_potential",
            "mean"
        ),
        gap_mean=(
            "ai_realization_gap",
            "mean"
        ),
        gap_median=(
            "ai_realization_gap",
            "median"
        ),
        gap_sd=(
            "ai_realization_gap",
            "std"
        )
    )
    .reset_index()
    .sort_values("year")
)

display(yearly.round(4))

In [ ]:
yearly.to_csv(
    TABLE_DIR /
    "Step_06_Yearly_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# FIGURE 1 — MEAN AI ADOPTION BY YEAR
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    yearly["year"],
    yearly["ai_adoption_mean"],
    marker="o"
)

ax.set_xlabel("Year")
ax.set_ylabel("Mean AI adoption (%)")
ax.set_title("Mean AI Adoption across EU-27")

# Show only observed years as integers (no 2021.5 ticks)
ax.set_xticks(yearly["year"].astype(int).tolist())
ax.set_xticklabels(yearly["year"].astype(int).astype(str).tolist())
ax.grid(alpha=0.25)

ax.set_xticks([2021, 2023, 2024, 2025])
ax.set_xticklabels(["2021", "2023", "2024", "2025"])

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure1_AI_Adoption_Trend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2 — MEAN AI POTENTIAL
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    yearly["year"],
    yearly["ai_potential_mean"],
    marker="o"
)

ax.set_xlabel("Year")
ax.set_ylabel("Mean AI Potential Index")
ax.set_title("Mean AI Potential across EU-27")

# Show only observed years as integers (no 2021.5 ticks)
ax.set_xticks(yearly["year"].astype(int).tolist())
ax.set_xticklabels(yearly["year"].astype(int).astype(str).tolist())
ax.grid(alpha=0.25)

ax.set_xticks([2021, 2023, 2024, 2025])
ax.set_xticklabels(["2021", "2023", "2024", "2025"])

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure2_AI_Potential_Trend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 3 — AI REALIZATION GAP TREND
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    yearly["year"],
    yearly["gap_mean"],
    marker="o"
)

ax.axhline(
    0,
    linewidth=1,
    linestyle="--"
)

ax.set_xlabel("Year")
ax.set_ylabel("Mean AI Realization Gap")
ax.set_title("AI Realization Gap across EU-27")

# Show only observed years as integers (no 2021.5 ticks)
ax.set_xticks(yearly["year"].astype(int).tolist())
ax.set_xticklabels(yearly["year"].astype(int).astype(str).tolist())
ax.grid(alpha=0.25)

ax.set_xticks([2021, 2023, 2024, 2025])
ax.set_xticklabels(["2021", "2023", "2024", "2025"])

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure3_AI_Realization_Gap_Trend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 4 — AI POTENTIAL VS AI ADOPTION
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    df["ai_potential"],
    df["ai_adoption_norm"],
    alpha=0.7
)

# 45-degree reference line
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

ax.set_xlabel("AI Potential Index")
ax.set_ylabel("Normalized AI Adoption")
ax.set_title("AI Potential and Realized AI Adoption")

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure4_Potential_vs_Adoption.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# COUNTRY-LEVEL AVERAGE GAP
# ============================================================

country_gap = (
    df
    .groupby(
        ["iso3", "country"]
    )
    .agg(
        mean_gap=(
            "ai_realization_gap",
            "mean"
        ),
        mean_potential=(
            "ai_potential",
            "mean"
        ),
        mean_adoption=(
            "ai_adoption_norm",
            "mean"
        )
    )
    .reset_index()
)

country_gap = country_gap.sort_values(
    "mean_gap",
    ascending=True
)

display(country_gap)

In [ ]:
# ============================================================
# FIGURE 5 — COUNTRY AVERAGE REALIZATION GAP
# ============================================================

fig, ax = plt.subplots(figsize=(10, 10))

ax.barh(
    country_gap["country"],
    country_gap["mean_gap"]
)

ax.axvline(
    0,
    linewidth=1,
    linestyle="--"
)

ax.set_xlabel("Mean AI Realization Gap")
ax.set_ylabel("")
ax.set_title(
    "Average AI Realization Gap by EU-27 Country"
)

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure5_Country_Average_Gap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# CHANGE IN GAP — 2021 TO 2025
# ============================================================

gap_change = (
    df[
        [
            "iso3",
            "country",
            "year",
            "ai_realization_gap"
        ]
    ]
    .pivot(
        index=[
            "iso3",
            "country"
        ],
        columns="year",
        values="ai_realization_gap"
    )
    .reset_index()
)

gap_change["change_2021_2025"] = (
    gap_change[2025]
    -
    gap_change[2021]
)

gap_change = gap_change.sort_values(
    "change_2021_2025"
)

display(gap_change)

In [ ]:
# ============================================================
# FIGURE 6 — GAP CHANGE 2021-2025
# ============================================================

fig, ax = plt.subplots(figsize=(10, 10))

ax.barh(
    gap_change["country"],
    gap_change["change_2021_2025"]
)

ax.axvline(
    0,
    linewidth=1,
    linestyle="--"
)

ax.set_xlabel(
    "Change in AI Realization Gap (2025 − 2021)"
)

ax.set_ylabel("")

ax.set_title(
    "Change in AI Realization Gap, 2021–2025"
)

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure6_Gap_Change_2021_2025.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# GAP ROBUSTNESS CORRELATIONS
# ============================================================

gap_corr = df[
    [
        "ai_realization_gap",
        "ai_realization_gap_z",
        "ai_realization_gap_equal"
    ]
].corr()

display(gap_corr.round(4))

In [ ]:
gap_corr.to_csv(
    TABLE_DIR /
    "Step_06_Gap_Robustness_Correlations.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# ENABLER CORRELATIONS
# ============================================================

enabler_corr = df[
    [
        "ai_realization_gap",
        "vhcn_pct",
        "ln_gdp_pc_pps"
    ]
].corr()

display(enabler_corr.round(4))

In [ ]:
# ============================================================
# FIGURE 7 — VHCN VS REALIZATION GAP
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    df["vhcn_pct"],
    df["ai_realization_gap"],
    alpha=0.7
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Fixed VHCN Coverage (%)")
ax.set_ylabel("AI Realization Gap")
ax.set_title(
    "Digital Infrastructure and AI Realization Gap"
)

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure7_VHCN_vs_Gap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 8 — GDP VS REALIZATION GAP
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    df["ln_gdp_pc_pps"],
    df["ai_realization_gap"],
    alpha=0.7
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Log GDP per Capita (PPS)")
ax.set_ylabel("AI Realization Gap")
ax.set_title(
    "Economic Capacity and AI Realization Gap"
)

plt.tight_layout()

fig.savefig(
    STEP06_FIGURE_DIR /
    "Step_06_Figure8_GDP_vs_Gap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# GAP DISTRIBUTION BY YEAR
# ============================================================

gap_quantiles = (
    df
    .groupby("year")
    ["ai_realization_gap"]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
    .reset_index()
)

gap_quantiles.columns = [
    "year",
    "p10",
    "p25",
    "median",
    "p75",
    "p90"
]

display(gap_quantiles.round(4))

In [ ]:
gap_quantiles.to_csv(
    TABLE_DIR /
    "Step_06_Gap_Quantiles_by_Year.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# SAVE CORE TABLES
# ============================================================

country_gap.to_csv(
    TABLE_DIR /
    "Step_06_Country_Average_Gap.csv",
    index=False,
    encoding="utf-8-sig"
)

gap_change.to_csv(
    TABLE_DIR /
    "Step_06_Gap_Change_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

enabler_corr.to_csv(
    TABLE_DIR /
    "Step_06_Enabler_Correlations.csv",
    encoding="utf-8-sig"
)

print("Step 06 tables saved.")

In [ ]:
# ============================================================
# STEP 06 FINAL REPORT
# ============================================================

print("=" * 72)
print("STEP 06 — DESCRIPTIVE ANALYSIS COMPLETED")
print("=" * 72)

print("Observations:", len(df))
print("Countries   :", df["iso3"].nunique())
print("Years       :", sorted(df["year"].unique()))

print("\nMean realization gap by year:")
display(
    yearly[
        [
            "year",
            "gap_mean",
            "gap_median"
        ]
    ].round(4)
)

print("\nRobustness correlations:")
display(
    gap_corr.round(4)
)

print("=" * 72)

In [ ]:
# ============================================================
# STEP 06 FIGURE FOLDER — VERIFY
# ============================================================
print("Figure output folder:", STEP06_FIGURE_DIR)
print("Saved PNG figures:", len(list(STEP06_FIGURE_DIR.glob("Step_06_Figure*.png"))))


In [ ]:
# ============================================================
# CHECK ALL EIGHT FIGURES
# ============================================================
expected_figures = [
    "Step_06_Figure1_AI_Adoption_Trend.png",
    "Step_06_Figure2_AI_Potential_Trend.png",
    "Step_06_Figure3_AI_Realization_Gap_Trend.png",
    "Step_06_Figure4_Potential_vs_Adoption.png",
    "Step_06_Figure5_Country_Average_Gap.png",
    "Step_06_Figure6_Gap_Change_2021_2025.png",
    "Step_06_Figure7_VHCN_vs_Gap.png",
    "Step_06_Figure8_GDP_vs_Gap.png",
]
missing_figures = [name for name in expected_figures if not (STEP06_FIGURE_DIR / name).exists()]
if missing_figures:
    print("Missing figures (run the figure cells above):", missing_figures)
else:
    print("All 8 figures saved successfully.")


In [ ]:
# ============================================================
# DOWNLOAD STEP 06 FIGURES AS ZIP
# ============================================================

import zipfile
from pathlib import Path
from google.colab import files

zip_path = Path(
    "/content/Step_06_Figures.zip"
)

missing = [name for name in expected_figures if not (STEP06_FIGURE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing required Step 06 figures: {missing}")

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for f in sorted(STEP06_FIGURE_DIR.glob("Step_06_Figure*.png")):
        z.write(
            f,
            arcname=f.name
        )

print("ZIP created:")
print(zip_path)

files.download(
    str(zip_path)
)

In [ ]:
# ============================================================
# DOWNLOAD STEP 06 TABLES AS ZIP
# ============================================================

from pathlib import Path
import zipfile
from google.colab import files

step06_table_files = [
    PROJECT_ROOT / "05_tables" / "Step_06_Table1_Descriptive_Statistics.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Yearly_Summary.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Gap_Robustness_Correlations.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Gap_Quantiles_by_Year.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Country_Average_Gap.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Gap_Change_2021_2025.csv",
    PROJECT_ROOT / "05_tables" / "Step_06_Enabler_Correlations.csv"
]

zip_path = Path("/content/Step_06_Tables.zip")

print("Files to include:\n")

missing_tables = [f for f in step06_table_files if not f.is_file()]
if missing_tables:
    raise FileNotFoundError("Missing required Step 06 tables: " + ", ".join(str(f) for f in missing_tables))

existing_files = []

for f in step06_table_files:
    exists = f.exists()
    print(f.name, "->", exists)

    if exists:
        existing_files.append(f)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for f in existing_files:
        z.write(
            f,
            arcname=f.name
        )

print("\nFiles included:", len(existing_files))
print("ZIP created:", zip_path)

files.download(str(zip_path))